In [1]:
from typing import Optional
from pymatgen.core.composition import Composition
import numpy as np


def is_empty_list(x):
    return isinstance(x, str) and x == "[]"

def try_parse_pressure(s):
    try:
        return float(s.split(" ")[0]) if isinstance(s, str) else None
    except Exception as e:
        print(f"Error processing '{s}': {e}")
        return None

def convert_mass_magnetization(
    comp: Composition, mass_magnetization: float | np.ndarray, cgs_to_si: bool = True
) -> float | np.ndarray:
    """
    转换质量磁化率的单位，根据布尔参数决定转换方向。
    支持 mass_magnetization 为标量或 numpy 数组的情况。

    参数:
        comp: pymatgen.core.composition.Composition
            物质的化学组成，用于计算摩尔质量 (g/mol)。
        mass_magnetization: float 或 numpy.ndarray
            - 当 cgs_to_si 为 True 时，表示 CGS 制的质量磁化率 (emu/g)；
            - 当 cgs_to_si 为 False 时，表示 SI 制的摩尔磁矩 (A·m²/mol)。
            可以是单个浮点数，也可以是 numpy 数组。
        cgs_to_si: bool, 默认 True
            转换方向选择：
                True  : 从 CGS (emu/g) 转换到 SI (A·m²/mol)
                False : 从 SI (A·m²/mol) 转换到 CGS (emu/g)

    返回:
        float 或 numpy.ndarray:
            转换后的磁化率数值。
            - 如果 cgs_to_si=True，单位为 A·m²/mol；
            - 如果 cgs_to_si=False，单位为 emu/g。
            如果输入是 numpy 数组，则返回相同形状的 numpy 数组；
            如果输入是标量，则返回标量浮点数。
    """
    # 计算摩尔质量 (g/mol)
    molar_mass = comp.weight  # g/mol

    # 将输入转为 numpy 数组以便支持数组运算
    mm_arr = np.asarray(mass_magnetization, dtype=float)

    if cgs_to_si:
        # 从 CGS 转换到 SI:
        # mass_magnetization (emu/g) * molar_mass (g/mol) = emu/mol
        # 1 emu = 1e-3 A·m², 所以转换为 SI 得到 A·m²/mol
        result_arr = mm_arr * molar_mass * 1e-3
    else:
        # 从 SI 转换到 CGS:
        # mass_magnetization 表示 SI 制 (A·m²/mol)
        # 先转换为 emu/mol: 1 A·m² = 1e3 emu, 然后除以 molar_mass (g/mol) 得到 emu/g
        result_arr = mm_arr * 1e3 / molar_mass

    # 如果输入是标量 (0 维数组)，将结果转回浮点数
    if result_arr.ndim == 0:
        return float(result_arr)
    return result_arr


def convert_composition(
    comp: str, require_computable_mass: bool = True, as_formula: bool = True
) -> str | Composition | None:
    try:
        c = Composition(comp)
        if len(c) == 0:
            return None
        if require_computable_mass:
            c.weight
        return c.formula if as_formula else c
    except Exception:
        return None


def format_title(text: str, to_snake: bool = False, skip_list: Optional[list[str]] = None) -> str:
    """
    格式化输入文本为首字母大写格式或蛇形命名法，并支持指定不需要格式化的单词。

    参数:
        text : str
            输入文本。
        to_snake : bool, 可选
            是否转换为蛇形命名法，默认为 False。
        skip_list : list[str], 可选
            指定部分单词不做格式化，默认为 None。

    返回:
        str
            格式化后的文本。
    """
    if not isinstance(text, str):
        raise ValueError("Input must be a string")

    if text is None or text.strip() == "":
        return text

    if skip_list is None:
        skip_list = []

    if to_snake:
        # 将空格拆分为单词，并转换为蛇形命名法，保留skip_list中的单词不变
        words = text.strip().split()
        formatted = [word if word in skip_list else word.lower() for word in words]
        return "_".join(formatted)
    else:
        # 将下划线替换为空格，并分割成单词
        words = text.replace("_", " ").strip().split()
        if not words:
            return text
        # 格式化：第一个单词首字母大写，其他单词全部小写（除非在skip_list中）
        formatted = []
        for i, word in enumerate(words):
            if word in skip_list:
                formatted.append(word)
            else:
                formatted.append(word.capitalize() if i == 0 else word.lower())
        return " ".join(formatted)

In [2]:
from datasets import load_dataset

ds = load_dataset("phonix-db/phonix-full")

/Users/liuchang/projects/foundation_model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
filtered_train = ds["train"].filter(
    lambda row: (
row["kp[W/mK]"] is not None
and row["klat[W/mK]"] is not None
and row["fc2_error[%]"] is not None
and row["fc3_error[%]"] is not None
and abs(row["fc2_error[%]"]) <= 10.0
and abs(row["fc3_error[%]"]) <= 10.0
and row["klat[W/mK]"] <= 2000
    )
)

filtered_train = filtered_train.to_pandas()[
    ['mp_id', 'formula', 'kp[W/mK]', 'kc[W/mK]', 'klat[W/mK]', 'fc3_error[%]']
].drop_duplicates(subset=['mp_id'])
filtered_train["formula"] = filtered_train["formula"].map(convert_composition)
filtered_train = filtered_train.rename(columns={
    'formula': 'composition',
})
filtered_train.to_parquet("../phonix-db-filtered_20260425.parquet", index=False)

filtered_train

,mp_id,composition,kp[W/mK],kc[W/mK],klat[W/mK],fc3_error[%]
0,mp-1000,Ba1 Te1,5.385133,0.033300,5.418433,0.210506
1,mp-147,Se1,1.416800,0.117300,1.534100,1.112320
3,mp-149,Si1,121.608400,0.026367,121.634767,0.065174
5,mp-154,N2,0.025600,0.184500,0.210100,0.262661
6,mp-160,B1,101.501700,0.238800,101.740500,0.370470
...,...,...,...,...,...,...
7428,mp-1390618,Sn1 O2,34.342267,0.088067,34.430333,0.564014
7429,mp-1391273,Zn1 Ni1 F6,4.018267,0.261333,4.279600,0.616551
7430,mp-1403065,Ti1 Zn1 F6,1.928867,0.406933,2.335800,0.703460
7431,mp-1423122,Rb2 Sn1 Te5,0.369767,0.088933,0.458700,0.668410


---

Process NEMAD data

In [4]:
import pandas as pd

superconductor = pd.read_csv("../NEMAD/superconductor_materials_unique.csv")
superconductor = superconductor[["Chemical_Composition", "Median_Tc_By_Composition_K", "Pressure", "Space_Group", "Experimental"]]
superconductor["Chemical_Composition"] = superconductor["Chemical_Composition"].map(convert_composition)
superconductor["Experimental"] = superconductor["Experimental"].apply(
    lambda x: (
        True
        if isinstance(x, str) and x.strip().strip("'").lower() == "true"
        else (
            False
            if isinstance(x, str) and x.strip().strip("'").lower() in {"false", "theoretical", "dft calculation"}
            else None
        )
    )
)
superconductor["Pressure"] = superconductor["Pressure"].apply(
    try_parse_pressure
)
superconductor = superconductor.dropna(subset=["Chemical_Composition", "Median_Tc_By_Composition_K", "Experimental"])

superconductor = superconductor.rename(columns={
    "Chemical_Composition": "composition",
    "Median_Tc_By_Composition_K": "Transition temperature[K]",
    "Pressure": "Pressure[GPa]",
    "Space_Group": "Space group",
    "Experimental": "Experimental"
})

superconductor.to_parquet("../NEMAD_superconductor_20260425.parquet", index=False)
superconductor

/Users/liuchang/projects/foundation_model/.venv/lib/python3.13/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)


Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing '0T': could not convert string to float: '0T'
Error processing '1% O2/Ar': could not convert string to float: '1%'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'Ambient': could not convert string to float: 'Ambient'
Error processing 'High pressure': could not convert string to float: 'High'
Error processing 'High pressure': could not convert string to float: 'High'
Error processing 'high pressure': could not convert string to float: 'high'
Error processing 'ambient': could not convert string to float: 'ambient'
Error processing 'ambient': could not convert string to 

/Users/liuchang/projects/foundation_model/.venv/lib/python3.13/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/Users/liuchang/projects/foundation_model/.venv/lib/python3.13/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)


,composition,Transition temperature[K],Pressure[GPa],Space group,Experimental
0,Ac1 B1 H16,39.00,100.0,Pm21b,False
1,Ac1 B1 H7,82.50,200.0,P-3m1,False
2,Ac1 B1 H8,44.50,200.0,P21/m,False
4,Ac1 B2 H14,42.00,200.0,Pnm21,False
5,Ac1 Be1 H10,165.00,300.0,Fm-3m,False
...,...,...,...,...,...
19050,Zr1 V2,8.70,NaN,NaN,True
19051,Zr33 V67,8.00,NaN,NaN,True
19052,Zr64 V36,1.30,NaN,NaN,True
19053,Zr1 W2,1.16,NaN,NaN,True


In [6]:
%run nemad_magnetic.py
import pandas as pd


magnetic = pd.read_csv("../NEMAD/magnetic_materials.csv")
magnetic = magnetic[
    [
        "Material_Name",
        "Magnetic_Moment",
        "Magnetization",
        "Curie",
        "Curie(Tc)",
        "Neel",
        "Neel(Tn)",
        "Space_Group",
        "Experimental",
    ]
]
magnetic["Material_Name"] = magnetic["Material_Name"].map(convert_composition)
magnetic = magnetic.dropna(subset=["Material_Name"]).drop_duplicates(
    subset=["Material_Name"]
)
magnetic = clean_magnetic_dataframe(magnetic)[['Material_Name', 'Magnetic_Moment', 'Magnetization', 'Curie', 'Neel', 'Space_Group', 'Experimental']]
magnetic = magnetic.rename(columns={
    "Material_Name": "composition",
    "Magnetic_Moment": "Magnetic moment[μB/f.u.]",
    "Magnetization": "Magnetization[A·m²/mol]",
    "Curie": "Curie temperature[K]",
    "Neel": "Neel temperature[K]",
    "Space_Group": "Space group",
    "Experimental": "Experimental"

})

magnetic.to_parquet("../NEMAD_magnetic_20260419.parquet", index=False)

magnetic

/var/folders/ms/trlx2qj16hg7p3qd0k66yc340000gn/T/ipykernel_38171/1371593955.py:5: DtypeWarning: Columns (10,15,16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  magnetic = pd.read_csv("../NEMAD/magnetic_materials.csv")


,composition,Magnetic moment[μB/f.u.],Magnetization[A·m²/mol],Curie temperature[K],Neel temperature[K],Space group,Experimental
0,Sr0.5 Nd0.5 Mn1 O3,NaN,NaN,NaN,50.0,NaN,True
1,Fe3 O4,NaN,NaN,NaN,NaN,NaN,True
2,Gd3 Si11 Pt23,NaN,NaN,42.00,NaN,Fm3̄m,True
3,Tb3 Si11 Pt23,NaN,NaN,25.00,NaN,Fm3̄m,True
4,Dy3 Si11 Pt23,NaN,NaN,15.99,NaN,Fm3̄m,True
...,...,...,...,...,...,...,...
33613,Lu2 Fe17 N2.7,NaN,NaN,NaN,NaN,NaN,True
33614,Lu2 Fe17 N2.5,NaN,NaN,NaN,NaN,NaN,True
33625,Pr1 B6,NaN,NaN,NaN,NaN,NaN,True
33632,Pr1 Al3 Pd2,NaN,NaN,NaN,NaN,NaN,True
